# RF-DETR — Appearance ReID tracking comparison (Colab)

Compare person tracking with appearance **ReID off vs on** over one clip.

The **same detections** feed two tracking pipelines that differ only in `reid_enabled`, so any change in track-id stability is attributable to ReID alone. The default lightweight ReID is a CPU-only HSV torso-color histogram (no extra dependencies); section 6 upgrades to a CNN embedding for grayscale/IR footage.

**How to read the results:** if `mean active` (average people per frame) stays about the same while `unique track ids` drops, ReID is reviving ids for people who left and returned. If the average count itself drops, ids are being wrongly merged — raise `--reid-similarity`.

> Tip: `Runtime → Change runtime type → GPU` before running (keypoint inference is slow on CPU).

In [ ]:
!nvidia-smi -L || echo 'No GPU detected — inference will be slow. Runtime > Change runtime type > GPU.'

## 1. Clone and install

In [ ]:
import os

REPO_URL = 'https://github.com/shingo257/rf-detr.git'
BRANCH = 'develop'

if not os.path.exists('rf-detr'):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd rf-detr
!pip -q install -e .
print('\nInstalled. If imports fail below, use Runtime > Restart session, then re-run from this cell (the clone is skipped automatically).')

## 2. Choose a video

Defaults to a public *people-walking* clip (people occlude each other — good for ReID). To use your own footage, uncomment the upload lines. The bundled `sample/*.mov` files are private and are **not** in the repo.

In [ ]:
import os
import cv2

VIDEO_URL = 'https://media.roboflow.com/supervision/video-examples/people-walking.mp4'
VIDEO = 'demo.mp4'
if not os.path.exists(VIDEO):
    !wget -q -O {VIDEO} {VIDEO_URL}

# --- To use your own video instead, uncomment (overrides the download above): ---
# from google.colab import files
# uploaded = files.upload()
# VIDEO = next(iter(uploaded))

cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f'Could not open {VIDEO} — re-run this cell or upload a valid video.'
print(f'Using {VIDEO}: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames, '
      f'{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
cap.release()

## 3. Run the comparison (ReID off vs on)

The first run downloads the keypoint model weights. `--max-frames` keeps it quick.

In [ ]:
!rfdetr-demo compare-reid --source {VIDEO} --max-frames 600 --json reid_metrics.json

import json
print('\n' + json.dumps(json.load(open('reid_metrics.json')), indent=2, ensure_ascii=False))

## 4. Sweep the thresholds

Vary the cost-blend `--reid-weight` and the revival `--reid-similarity`. `weight=0.0` isolates gallery revival. Pick the **highest** `similarity` that still consolidates ids (`ids_on` well below `ids_off`) **while** `mean_on` stays close to `mean_off` — that is the setting most resistant to merging different people.

In [ ]:
import json, subprocess
import pandas as pd

rows = []
for weight in [0.0, 0.3, 0.6]:
    for similarity in [0.5, 0.7, 0.8, 0.9, 0.95]:
        subprocess.run(
            ['rfdetr-demo', 'compare-reid', '--source', VIDEO, '--max-frames', '600',
             '--reid-weight', str(weight), '--reid-similarity', str(similarity), '--json', 'sweep.json'],
            check=True,
        )
        data = json.load(open('sweep.json'))
        off, on = data['reid_off'], data['reid_on']
        rows.append({'weight': weight, 'similarity': similarity,
                     'ids_off': off['unique_ids'], 'ids_on': on['unique_ids'],
                     'mean_off': round(off['mean_active'], 2), 'mean_on': round(on['mean_active'], 2),
                     'std_on': round(on['count_std'], 2)})

pd.DataFrame(rows)

## 5. Visual check — id-labeled OFF vs ON

Render both videos from the **same detections** with track ids drawn (number by each box; trailing `*` = held through an occlusion). Set `--reid-similarity` to the value you picked.

Clips are written to `compare_out/` on the Colab VM (ephemeral, **not** Google Drive) and transcoded to H.264 for inline playback. To keep them: `from google.colab import files; files.download('compare_out/reid_on.mp4')`.

**What to look for:**
- Good: in **ON**, a person keeps the same id/color across a brief occlusion where **OFF** flips them to a new id.
- Over-merge: in **ON**, two clearly different people share one id/color -> raise `--reid-similarity`.

In [ ]:
!rfdetr-demo compare-reid --source {VIDEO} --max-frames 600 --reid-similarity 0.85 --write-video --out-dir compare_out

import os
from base64 import b64encode
from IPython.display import HTML, display

def show(src, label):
    dst = src.replace('.mp4', '_h264.mp4')
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    mp4 = b64encode(open(dst, 'rb').read()).decode()
    display(HTML(
        f'<p><b>{label}</b> &mdash; number = track id, <code>*</code> = held during occlusion</p>'
        f'<video width=640 controls><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>'
    ))

show('compare_out/reid_off.mp4', 'ReID OFF')
show('compare_out/reid_on.mp4', 'ReID ON (similarity 0.85)')

## 6. Stronger ReID with an ONNX embedding

The color histogram is weak on **grayscale/IR** footage or when everyone wears similar colors. A CNN **embedding** is robust to both. The next cell exports a small MobileNetV3 feature extractor to `reid.onnx` using torchvision weights (always downloadable), then runs the comparison with `--reid-backend embedding`.

Model contract: input `1x3x256x128` (RGB, ImageNet-normalized); output one embedding vector (L2-normalized internally). The embedding uses **cosine** similarity, so start `--reid-similarity` around **0.5-0.7**.

For best quality use a **person-ReID-trained** model (OSNet). If torchreid installs cleanly, replace the export cell with:
```python
!pip -q install git+https://github.com/KaiyangZhou/deep-person-reid.git
import torch, torchreid
m = torchreid.models.build_model('osnet_x0_25', num_classes=1000, pretrained=True).eval()
torch.onnx.export(m, torch.randn(1,3,256,128), 'reid.onnx',
                  input_names=['input'], output_names=['output'], opset_version=18)
```

In [ ]:
# torch.onnx.export on Colab's torch needs onnxscript for the default exporter.
!pip -q install onnx onnxscript

# Reliable export: MobileNetV3 features -> reid.onnx (torchvision weights always download).
import torch
import torch.nn as nn
import torchvision

backbone = torchvision.models.mobilenet_v3_small(weights='DEFAULT')
extractor = nn.Sequential(backbone.features, nn.AdaptiveAvgPool2d(1), nn.Flatten()).eval()
with torch.no_grad():
    dim = int(extractor(torch.randn(1, 3, 256, 128)).shape[1])
torch.onnx.export(extractor, torch.randn(1, 3, 256, 128), 'reid.onnx',
                  input_names=['input'], output_names=['output'], opset_version=18)
print(f'wrote reid.onnx (embedding dim {dim})')

In [ ]:
!pip -q install -e '.[reid]'
!rfdetr-demo compare-reid --source {VIDEO} --max-frames 600 \
    --reid-backend embedding --reid-model reid.onnx --reid-similarity 0.6 --json reid_embed.json

import json
print(json.dumps(json.load(open('reid_embed.json')), indent=2, ensure_ascii=False))

## 7. Detection recall (how many people are found)

Tracking can only follow **detected** people; in dense/top-down scenes many are missed. Two levers on `compare-reid`:
- `--threshold` — lower detects more (and more false positives).
- `--resolution` — higher detects smaller/distant people (slower). It must satisfy the model's patch divisibility; the cell below tries values and **skips** any the model rejects.

Watch `mean_people` / `max_people` (the ReID-off detection counts) climb as recall improves. This is upstream of tracking and is usually the biggest lever for person-count accuracy.

In [ ]:
import json, subprocess
import pandas as pd

rows = []
for threshold in [0.5, 0.3, 0.2]:
    for resolution in [None, 768, 960]:   # multiples of patch_size*4 = 48 (default is 576); rejects are skipped
        cmd = ['rfdetr-demo', 'compare-reid', '--source', VIDEO, '--max-frames', '100',
               '--threshold', str(threshold), '--json', 'recall.json']
        if resolution is not None:
            cmd += ['--resolution', str(resolution)]
        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as error:
            print(f'thr={threshold} res={resolution}: rejected ({error.stderr.strip().splitlines()[-1] if error.stderr else error})')
            continue
        off = json.load(open('recall.json'))['reid_off']
        row = {'threshold': threshold, 'resolution': resolution or 'default (576)',
               'mean_people': round(off['mean_active'], 2), 'max_people': off['max_active']}
        rows.append(row)
        print(row)

pd.DataFrame(rows)

## 8. Detection-based tracking + counting (two-stage, Stage 1)

The person-**detection** model finds far more people than the keypoint model (≈28 vs ≈20 on this clip) and is faster. `--track` runs those detections through the same box-IoU tracker (motion prediction, gate, and appearance ReID all apply), drawing stable ids and a live count. This is the recommended pipeline for **person counting / flow**; add pose on a subset later (Stage 2).

ReID knobs still apply via env vars (e.g. `RFDETR_TRACK_REID=1`).

In [ ]:
!rfdetr-demo video --task detect --person-only --track --model large \
    --resolution 960 --threshold 0.25 --source {VIDEO} --max-frames 200 --output track.mp4

import os
from base64 import b64encode
from IPython.display import HTML, display

os.system('ffmpeg -y -loglevel error -i track.mp4 -vcodec libx264 -pix_fmt yuv420p track_h264.mp4')
display(HTML(f'<video width=760 controls><source src="data:video/mp4;base64,'
             f'{b64encode(open("track_h264.mp4", "rb").read()).decode()}" type="video/mp4"></video>'))

## Tuning cheatsheet

All default off; enable per run via env vars or the `compare-reid` flags.

| Env var | Flag | Meaning | Default |
| --- | --- | --- | --- |
| `RFDETR_TRACK_REID` | (compare-reid always runs both) | Enable appearance ReID | off |
| `RFDETR_REID_BACKEND` | `--reid-backend` | `histogram` (color, no deps) or `embedding` (ONNX, grayscale-robust) | histogram |
| `RFDETR_REID_MODEL` | `--reid-model` | ONNX ReID model path (embedding backend) | — |
| `RFDETR_REID_WEIGHT` | `--reid-weight` | Appearance vs IoU cost blend (0..1) | 0.3 |
| `RFDETR_REID_SIMILARITY` | `--reid-similarity` | Min match to revive an id (intersection or cosine) | 0.5 |
| `RFDETR_REID_GALLERY_FRAMES` | `--reid-gallery-frames` | How long a retired id stays revivable | 60 |
| `RFDETR_REID_EMA` | — | Descriptor smoothing (0..1) | 0.9 |
| — | `--threshold` | Detection threshold (lower = more recall) | 0.5 |
| — | `--resolution` | Keypoint model input resolution (higher = smaller people) | model default |

The histogram backend is CPU-only and dependency-free but weak on grayscale/IR footage. Prefer the embedding backend (section 6) for surveillance/IR feeds. For person-count accuracy, detection recall (section 7) is usually the bigger lever than tracking.